# Benchmark: 20 Shifted Classical Functions (Cleaned)

## Purpose
Tutorial notebook comparing **HPPSO** with PSO variants, GA-MPC, GWO, SHADE, CMA-ES, Sep-CMA-ES, and CSA on 20 shifted benchmarks.

## Modes
- **`USE_SAVED_RESULTS = True`** (default): loads pre-computed pickles from `results/` — no optimization runs.
- **`USE_SAVED_RESULTS = False`**: runs a **small** demo benchmark (few runs/iterations). Increase settings only when you intend to re-run experiments.

## Inputs
- `results/all_convergence_histories 30d.pkl`, `best_costs_30d.pkl`, CMA-ES merge files (see reproduction pipeline).

## Outputs
- Summary tables and optional convergence plots in `reproduced_figures/`.
- Aligns with paper **Tables 2, 4, 6** and **Figure 7** (30D).


## 1. Imports and configuration


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "nb_helpers.py").exists():
    sys.path.insert(0, str(NOTEBOOK_DIR))
elif (NOTEBOOK_DIR.parent / "notebooks" / "nb_helpers.py").exists():
    sys.path.insert(0, str(NOTEBOOK_DIR.parent / "notebooks"))
REPO_ROOT = NOTEBOOK_DIR.parent if (NOTEBOOK_DIR / "nb_helpers.py").exists() else NOTEBOOK_DIR.parent.parent

import nb_helpers as nh
nh.ensure_reproduction_imports()

from reproduction.config import FUNCTION_NAMES, DIMENSIONS
from reproduction.visualize import plot_hppso_median_panels, plot_all_algorithms_convergence
from reproduction.tables import save_tables
from hppso.benchmarks.classical import build_problem_suite
from hppso.experiments.runners import DEFAULT_ALGORITHMS, run_benchmark
from hppso.utils.statistics import overall_average_ranks

plt.rcParams.update(nh.PLOT_STYLE)

# --- Experiment settings (paper: 30D, 20 runs, pop=30, 500 iters) ---
DIM = 30
USE_SAVED_RESULTS = True  # set False only to run a lightweight demo

# Demo-only settings (ignored when USE_SAVED_RESULTS=True)
NUM_RUNS = 5
POP_SIZE = 30
MAX_ITERS = 200


## 2. Load pre-computed results


In [ ]:
if USE_SAVED_RESULTS:
    dataset = nh.load_merged(DIM)
    print(f"Loaded merged {DIM}D data — algorithms: {', '.join(dataset.algorithms())}")
    print("Result pickles:", ", ".join(nh.list_result_files()[:4]), "...")
else:
    dataset = None
    print("Will run live benchmark in next section (demo settings).")


## 3. Live benchmark (optional demo only)


In [ ]:
if not USE_SAVED_RESULTS:
    np.random.seed(42)
    problems = build_problem_suite(dim=DIM, random_shift=False)
    results = run_benchmark(
        DEFAULT_ALGORITHMS, problems,
        num_runs=NUM_RUNS, pop_size=POP_SIZE, max_iters=MAX_ITERS,
    )
    rows = []
    for algo, prob_results in results.items():
        for func, data in prob_results.items():
            rows.append({
                "Function": func, "Algorithm": algo,
                "Mean Best Score": data["avg_final_score"],
                "Std Best Score": np.nanstd(data["all_final_scores"]),
            })
    df = pd.DataFrame(rows)
else:
    raw = nh.best_costs_to_dataframe(dataset)
    df = (
        raw.groupby(["function", "algorithm"], as_index=False)
        .agg(Mean_Best_Score=("best_cost", "mean"), Std_Best_Score=("best_cost", "std"))
        .rename(columns={"function": "Function", "algorithm": "Algorithm", "Mean_Best_Score": "Mean Best Score", "Std_Best_Score": "Std Best Score"})
    )


## 4. Analysis — performance table and ranks


In [ ]:
display_df = df.rename(columns={"function": "Function", "algorithm": "Algorithm"}) if "function" in df.columns else df
pivot = display_df.pivot_table(index="Function", columns="Algorithm", values="Mean Best Score")
display(pivot)

rank_input = display_df.rename(columns={"Function": "function", "Algorithm": "algorithm", "Mean Best Score": "Mean Best Score"})
print("\nOverall average ranks (lower is better):")
print(overall_average_ranks(rank_input))


## 5. Export tables (reproduced_tables/)


In [ ]:
if USE_SAVED_RESULTS:
    paths = save_tables(dataset)
    for k, p in paths.items():
        print(f"  {k}: {p.name}")


## 6. Visualization


In [ ]:
if USE_SAVED_RESULTS:
    out = nh.REPRODUCED_FIGURES
    out.mkdir(exist_ok=True)
    fig7 = plot_hppso_median_panels(dataset, figure_number=7, output_dir=out)
    print(f"Figure 7: {fig7.name}")
    plot_all_algorithms_convergence(dataset, "f1_sphere", output_dir=out / "supplementary")
    print("Supplementary convergence plot for f1_sphere saved.")


## Notes
- CMA-ES 30D results are merged from `all_convergence_histories 30d cmaes only.pkl` (see `reproduction/merge.py`).
- Shift vectors: `reproduced_tables/shift_vectors_30D.csv` or `results/all_svs30d.pkl`.
- Full pipeline: `python -m reproduction.run_reproduction`
